# Phase 5 Cross-Seed Cue-Regime Sweep — K=1, β=10 (Colab, Drive-resumable)

**Active phase:** 5
**Purpose:** Cue-regime sweep at the confirmed-best operating point (β=10, K=1, γ=0.5 per [report 057](https://github.com/Dypatterson/Neuro-AI/blob/main/reports/057_phase5_cross_seed_beta_sweep_K1.md)). The grid is `binding_noise_std × content_distortion`; for each cell, the aggregator records paired ΔE plus basin-membership and ordering counts.

**Locked scope:**
- Fixed `β=10.0`, `γ=0.5`, `K=1`.
- Executable grid is the pre-committed listed grid: `binding_noise_std ∈ {0.01, 0.05, 0.10, 0.20}` × `content_distortion ∈ {0.0, 0.2, 0.4, 0.6, 0.8, 1.0}` = 24 cells per seed. Some older prose calls this a 36-cell sweep; do not add unlisted cells mid-run to reconcile the count.
- `n_cues=100`, CUDA, ten A1' seed snapshots.
- Local runs are profile/debug only; the full sweep is Colab + Drive-resume only.

**Per-cell metrics:**
- `mean ΔE_raw` + 95% CI + seeds positive
- `ΔE / 5.5e-3 floor` ratio
- Ordering counts: `role < content < random`, `role < content`, `random_lowest` (the pathology metric — report 057 found random produces lowest energy at 9/10 seeds with K=1, β=10)
- Per-condition basin hit rate (argmax similarity == role_target_idx?)
- Per-condition mean role-target rank (1-indexed in similarity ordering)
- Per-condition entropy / max_w / mean energy

**Execution branch:** run the one-cell profile first. If it is slow (>30 sec), inspect known hot spots before launching the full sweep. If it is fast but parallel workers stall, run the full sweep one seed at a time with Drive-backed per-seed JSONs. If sequential Colab still fails, stop and treat harness/runtime reliability as the blocker before interpreting cue-regime results.

**Decision discipline:** this is a drill-down, not a graduation experiment. No cue-regime cell, even one above the magnitude floor, authorizes treating those cue settings as a graduation operating point. Evidence informs all four live strategic options: lower-D redesign; advance with an explicit sub-floor caveat (currently contraindicated); successor headline/reformulation; and basin-shape priors. No retuning of β/K/γ/ε/τ/formulation.


In [ ]:
# 1. Clone repo at the patched commit (must include Drive-resumable cue sweep mode).
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI

# Set this to the branch or commit that contains this notebook + aggregator patch
# if it has not landed on main yet.
GIT_REF = 'main'
!git checkout {GIT_REF}
!git log --oneline -6

import subprocess, sys
markers = [
    ('--cue-regime-sweep',             'scripts/phase5_frozen_snapshot_audit.py', 'harness cue-regime mode'),
    ('_run_headline_cue_regime_sweep', 'scripts/phase5_frozen_snapshot_audit.py', 'cue-regime sweep core'),
    ('_basin_diagnostics',             'scripts/phase5_frozen_snapshot_audit.py', 'basin diagnostics helper'),
    ('--expected-seeds',               'scripts/aggregate_cue_sweep.py',          'seed gap reporting'),
    ('No cue-regime cell',             'scripts/aggregate_cue_sweep.py',          'drill-down-only wording'),
]
for marker, fpath, label in markers:
    r = subprocess.run(['grep', '-n', '-e', marker, fpath], capture_output=True, text=True)
    status = 'OK' if r.returncode == 0 else 'MISSING'
    print(f'  [{status}] {label}: {marker} in {fpath}')
    if r.returncode != 0:
        sys.exit(1)


In [ ]:
# 2. Mount Drive.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RESULTS = '/content/drive/MyDrive/neuro-ai/results'
print('drive results root:', DRIVE_RESULTS)


In [ ]:
# 3. Locate the 10 A1' snapshots.
import os
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
DRIVE_RESULTS = '/content/drive/MyDrive/neuro-ai/results'

snapshot_paths = {}
missing = []
for s in SEEDS:
    p = f'{DRIVE_RESULTS}/phase5_headline_substrate_seed{s}/snapshots/phase3_phase4_w4_step1800.pt'
    if os.path.exists(p):
        snapshot_paths[s] = p
        print(f'  seed {s:>3}: ok')
    else:
        missing.append(s)
        print(f'  seed {s:>3}: MISSING at {p}')

if missing:
    print(f'\n!!! missing: {missing}')


In [ ]:
# 4. Parent-CPU sanity. Do NOT init CUDA in parent.
!nvidia-smi --query-gpu=name,memory.total --format=csv | head -3


In [ ]:
# 5. Profile one cue-regime cell and verify Drive-resumable atomic write.
import json, os, shutil, subprocess, time
from pathlib import Path

RUN_TAG = 'phase5_cross_seed_cue_sweep'
PROFILE_TAG = f'{RUN_TAG}_profile'
N_CUES = 100
K_MAIN = 1
GAMMA = 0.5
BETA = 10.0
BNS_GRID = '0.01,0.05,0.10,0.20'
CD_GRID = '0.0,0.2,0.4,0.6,0.8,1.0'
GRID_CELL_COUNT = len(BNS_GRID.split(',')) * len(CD_GRID.split(','))
PROFILE_SEED = 17
PROFILE_BNS = '0.05'
PROFILE_CD = '0.0'
PROFILE_CELL_COUNT = 1

assert K_MAIN == 1 and GAMMA == 0.5 and BETA == 10.0
assert GRID_CELL_COUNT == 24, GRID_CELL_COUNT
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'

work_root = Path(f'/content/{RUN_TAG}_work')
profile_local_root = work_root / 'profile'
drive_root = Path(DRIVE_RESULTS) / RUN_TAG
drive_profile_root = drive_root / 'profile'
drive_log_root = drive_root / 'logs'
for p in (profile_local_root, drive_profile_root, drive_log_root):
    p.mkdir(parents=True, exist_ok=True)


def validate_cue_json(path, expected_cells=None):
    with open(path) as f:
        data = json.load(f)
    sweep = data.get('cue_regime_sweep')
    if not isinstance(sweep, dict) or not isinstance(sweep.get('cells'), list):
        raise RuntimeError(f'{path} is not a cue-regime sweep JSON')
    if expected_cells is not None and len(sweep['cells']) != expected_cells:
        raise RuntimeError(
            f'{path} has {len(sweep["cells"])} cells; expected {expected_cells}'
        )
    return data


def copy_file_to_drive_complete(local_path, drive_path, *, remove_local=False):
    local_path = Path(local_path)
    drive_path = Path(drive_path)
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_drive_path = drive_path.with_name(drive_path.name + '.partial')
    if tmp_drive_path.exists():
        tmp_drive_path.unlink()
    if remove_local:
        shutil.move(str(local_path), str(tmp_drive_path))
    else:
        shutil.copy2(local_path, tmp_drive_path)
    os.replace(tmp_drive_path, drive_path)
    return drive_path


def run_audit(seed, snapshot, out_path, log_path, binding_noise, content_distortion, expected_cells):
    out_path = Path(out_path)
    log_path = Path(log_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        out_path.unlink()
    cmd = [
        'python', 'scripts/phase5_frozen_snapshot_audit.py',
        '--snapshot', str(snapshot),
        '--output', str(out_path),
        '--cue-regime-sweep',
        '--cue-regime-binding-noise', str(binding_noise),
        '--cue-regime-content-distortion', str(content_distortion),
        '--cue-regime-beta', str(BETA),
        '--cue-regime-n-cues', str(N_CUES),
        '--headline-k-main', str(K_MAIN),
        '--headline-gamma', str(GAMMA),
        '--device', 'cuda',
    ]
    t0 = time.time()
    with open(log_path, 'w') as logf:
        r = subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT, env=os.environ.copy(), text=True)
    elapsed = time.time() - t0
    if r.returncode != 0:
        tail = log_path.read_text(errors='replace').splitlines()[-40:]
        print('\n'.join(tail))
        raise RuntimeError(f'seed {seed} failed rc={r.returncode}; see {log_path}')
    data = validate_cue_json(out_path, expected_cells=expected_cells)
    return elapsed, data

profile_drive_json = drive_profile_root / f'profile_seed{PROFILE_SEED}_bns{PROFILE_BNS}_cd{PROFILE_CD}.json'
profile_drive_log = drive_profile_root / f'profile_seed{PROFILE_SEED}_bns{PROFILE_BNS}_cd{PROFILE_CD}.log'
profile_elapsed = None

if profile_drive_json.exists():
    validate_cue_json(profile_drive_json, expected_cells=PROFILE_CELL_COUNT)
    print(f'Drive resume check passed: {profile_drive_json} already exists and is valid')
else:
    if PROFILE_SEED not in snapshot_paths:
        raise RuntimeError(f'profile seed {PROFILE_SEED} snapshot missing; cannot profile')
    local_json = profile_local_root / profile_drive_json.name
    local_log = profile_local_root / profile_drive_log.name
    profile_elapsed, profile_data = run_audit(
        PROFILE_SEED,
        snapshot_paths[PROFILE_SEED],
        local_json,
        local_log,
        PROFILE_BNS,
        PROFILE_CD,
        PROFILE_CELL_COUNT,
    )
    copy_file_to_drive_complete(local_json, profile_drive_json, remove_local=True)
    copy_file_to_drive_complete(local_log, profile_drive_log, remove_local=False)
    validate_cue_json(profile_drive_json, expected_cells=PROFILE_CELL_COUNT)
    print(f'profile cell completed in {profile_elapsed:.1f}s and published to Drive: {profile_drive_json}')

print('\nExecution branch:')
if profile_elapsed is None:
    print('- Profile output already existed on Drive; resume behavior is verified without recompute.')
    print('- Delete the profile JSON if you need a fresh wall-clock measurement.')
elif profile_elapsed > 30:
    print('- Slow profile (>30s): inspect compute_branch_diagnostics for redundant _unbiased_energy calls.')
    print('- Also inspect repeated similarity_matrix calls in basin diagnostics and batch similarity work before full sweep.')
elif profile_elapsed <= 10:
    print('- Fast profile (<=10s): harness is probably fine; if parallel workers stall, run one seed at a time.')
else:
    print('- Moderate profile: default to the sequential Drive-backed sweep unless you have a strong reason to parallelize.')


In [ ]:
# 6. Full cross-seed cue-regime sweep, one seed at a time with Drive resume.
import os, signal, time
from pathlib import Path

seed_local_root = work_root / 'seeds'
seed_log_root = work_root / 'logs'
for p in (seed_local_root, seed_log_root, drive_root, drive_log_root):
    p.mkdir(parents=True, exist_ok=True)

# Defensive: kill any survivors from a previous interrupted Colab run.
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'phase5_frozen_snapshot_audit' in line and 'grep' not in line:
        try:
            os.kill(int(line.split()[0]), signal.SIGKILL)
            killed += 1
        except Exception:
            pass
if killed:
    print(f'killed {killed} survivor processes')
    time.sleep(2)

# Resume from Drive final JSONs only. Partial files are ignored by design.
valid_done = []
for seed in snapshot_paths:
    drive_json = drive_root / f'seed{seed}.json'
    if drive_json.exists():
        validate_cue_json(drive_json, expected_cells=GRID_CELL_COUNT)
        valid_done.append(seed)

remaining = [s for s in snapshot_paths if s not in valid_done]
print(f'Drive already has {len(valid_done)} valid seed JSONs: {valid_done}')
print(f'{len(remaining)} seeds to run sequentially: {remaining}')


def summarize_seed(seed, data):
    cells = data['cue_regime_sweep']['cells']
    best_dE = max(cells, key=lambda c: c['mean_delta_e_raw'])
    best_hit = max(cells, key=lambda c: c['per_condition_basin_hit_rate']['role'])
    return (
        f'seed {seed} DONE — best ΔE={best_dE["mean_delta_e_raw"]:+.5f} '
        f'(bns={best_dE["binding_noise_std"]}, cd={best_dE["content_distortion"]}); '
        f'best hit_role={best_hit["per_condition_basin_hit_rate"]["role"]:.2f} '
        f'at bns={best_hit["binding_noise_std"]}, cd={best_hit["content_distortion"]}'
    )

failed_seeds = []
t0 = time.time()
for i, seed in enumerate(remaining, start=1):
    print(f'\n[{i}/{len(remaining)}] running seed {seed}')
    local_json = seed_local_root / f'seed{seed}.json'
    local_log = seed_log_root / f'seed{seed}.log'
    drive_json = drive_root / f'seed{seed}.json'
    drive_log = drive_log_root / f'seed{seed}.log'
    try:
        elapsed, data = run_audit(
            seed,
            snapshot_paths[seed],
            local_json,
            local_log,
            BNS_GRID,
            CD_GRID,
            GRID_CELL_COUNT,
        )
        copy_file_to_drive_complete(local_json, drive_json, remove_local=True)
        copy_file_to_drive_complete(local_log, drive_log, remove_local=False)
        validate_cue_json(drive_json, expected_cells=GRID_CELL_COUNT)
        print(f'  {summarize_seed(seed, data)}; elapsed={elapsed/60:.1f} min')
    except Exception as exc:
        failed_seeds.append(seed)
        print(f'  seed {seed} FAILED: {exc}')
        break

if failed_seeds:
    raise RuntimeError(
        f'Sequential Colab failed for seeds {failed_seeds}; stop and treat harness/runtime reliability as the blocker.'
    )

per_seed_jsons = {
    s: drive_root / f'seed{s}.json'
    for s in snapshot_paths
    if (drive_root / f'seed{s}.json').exists()
}
print(f'\n[total] {(time.time() - t0)/60:.1f} min this cell; per_seed_jsons covers {len(per_seed_jsons)} seeds: {sorted(per_seed_jsons)}')


In [ ]:
# 7. Run cross-seed aggregator over Drive-backed seed JSONs.
import subprocess, os, json
from pathlib import Path

env = {**os.environ, 'PYTHONPATH': '/content/Neuro-AI/src'}
aggregate_local_root = work_root / 'aggregate'
aggregate_local_root.mkdir(parents=True, exist_ok=True)
agg_local_path = aggregate_local_root / 'cross_seed_aggregate.json'
agg_local_md = agg_local_path.with_suffix('.md')
agg_drive_path = drive_root / 'cross_seed_aggregate.json'
agg_drive_md = drive_root / 'cross_seed_aggregate.md'
expected_seed_arg = ','.join(str(s) for s in SEEDS)

r = subprocess.run([
    'python', 'scripts/aggregate_cue_sweep.py',
    '--sweep-root', str(drive_root),
    '--output', str(agg_local_path),
    '--expected-seeds', expected_seed_arg,
], env=env, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError(f'aggregator failed rc={r.returncode}')

with open(agg_local_path) as f:
    aggregate = json.load(f)
if len(aggregate.get('cells', [])) != GRID_CELL_COUNT:
    raise RuntimeError(f'aggregate has {len(aggregate.get("cells", []))} cells; expected {GRID_CELL_COUNT}')

copy_file_to_drive_complete(agg_local_path, agg_drive_path, remove_local=False)
copy_file_to_drive_complete(agg_local_md, agg_drive_md, remove_local=False)
print(f'published aggregate JSON + markdown to {drive_root}')


In [ ]:
# 8. Print the aggregator's markdown from Drive.
md_path = drive_root / 'cross_seed_aggregate.md'
print(md_path.read_text())


In [ ]:
# 9. List Drive outputs.
for p in sorted(drive_root.rglob('*')):
    if p.is_file():
        print(p)
